![FloPy: The Next Generation](../../images/flopy_nextgen.png)

A perennial predicament: does this FloPy work with that MF6?

We begin with some Jupyter magic necessary to hot-reload `flopy4` after regenerating some modules below. Note the warning that appears.

In [ ]:
%load_ext autoreload
%autoreload 1
%aimport flopy4

The suggested command would try to sync `flopy4` to the MF6 discovered on the `PATH`, falling back to the latest MF6 release if an MF6 executable can't be found or if the executable is not an official release version. Since we have a local development executable, `flopy4` would not be able to automatically identify compatible DFNs. Instead, we can pass a local path to the definition files.

In [ ]:
!flopy4 mf6 sync ../../modflow6/doc/mf6io/mf6ivar/dfn

In [ ]:
!flopy4 mf6 status

Hot-reload the `flopy4.mf6` module...

In [ ]:
import sys
import importlib
import flopy4.mf6 as mf6
importlib.reload(sys.modules['flopy4.mf6._contract'])
mf6 = importlib.reload(mf6)

The `flopy4.mf6` module is stamped with the `MF6_VERSION` used to generate it.

In [ ]:
mf6.MF6_VERSION

In the intro slides, we mentioned "code as documentation". FloPy3 already supports this in some capacity, but not everywhere.

In [ ]:
from flopy.mf6 import ModflowGwf, ModflowGwfdis

ModflowGwf.dfn
ModflowGwfdis.dfn

But this isn't always easy to interpret in a Python context. And DFNs contain some information that's relevant mainly to the MF6 input file format. This is arguably a serialization concern &mdash; especially now that MF6 has begun to support alternative input formats, like NetCDF.

In flopy4 we're aiming to keep the Python objects agnostic to input file format details and easily introspectable, so you can easily see how to interact with components of an MF6 simulation, without needing to think about how they will be translated into MF6 input files.

We can use the `fields_dict` function to inspect component classes programmatically. We can also look at the class definition itself.

In [ ]:
from flopy4.mf6.gwf import Dis
from flopy4.mf6.spec import fields_dict

dis_fields = fields_dict(Dis)
list(dis_fields.keys())

In the intro slides, we mentioned Deltares' `imod-python`, from which we've taken a lot of inspiration. They've come up with some serious user experience improvements. For example, it's natural to build a time discretization from timestamps:

In [ ]:
from flopy4.mf6.utils.time import Time

time = Time.from_timestamps(["2020-01-01", "2020-01-05", "2020-01-15"], nstp=5, tsmult=1.2)
time.time_units
"Period lengths: " + ", ".join([f"{pl} {time.time_units}" for pl in time.perlen])

This can then be converted to the format expected by an MF6 TDIS package.

In [ ]:
time.perioddata

Suppose you want to start building a package before building the model or simulation it will eventually be a part of.

In [ ]:
from flopy.mf6 import MFSimulation, ModflowGwf, ModflowGwfchdg

fp3_chdg = ModflowGwfchdg(head=1.1)

In `flopy`, package initialization expects a model parameter, and model initialization expects a simulation parameter. In other words, simulations must be built from the top down.

In [ ]:
fp3_sim = MFSimulation()
fp3_gwf = ModflowGwf(fp3_sim)
fp3_chdg = ModflowGwfchdg(fp3_gwf, head=1.1)

In `flopy4`, parent components are not required. The only requirement is that array dimensions be provided.

In [ ]:
from flopy4.mf6.utils.grid import StructuredGrid
from flopy4.mf6.gwf import Chdg

grid = StructuredGrid(nlay=1, nrow=2, ncol=2)
dims = {
    "nper": time.nper,
    "nodes": grid.nnodes,
}
chdg = Chdg(dims=dims, head=1.1)

Dimensions relevant to each package are available for inspection.

In [ ]:
chdg.dims

This comes courtesy of `xarray`. To get an `xarray.Dataset` view of a package:

In [ ]:
chdg.to_xarray()

This is a good opportunity to show a few different syntax variants for initializing fields. We've been showing the "grid-array" variant of the CHD package so far, which is meant for dense array data. The standard CHD package is ideal for sparser data.

In [ ]:
from flopy4.mf6.gwf import Chd

chd = Chd(dims=dims, head={"*": 1.1})  # "*" means "apply to all stress periods"
chd = Chd(dims=dims, head={0: 1.1, 1: 1.2})  # dict keyed by stress period number

Another common `flopy` friction point is interacting with package attributes. Let's set up a DIS package and access the `nlay` field.

In [ ]:
from flopy.mf6 import MFSimulation, ModflowGwf, ModflowGwfdis

fp3_sim = MFSimulation()
fp3_gwf = ModflowGwf(fp3_sim)
fp3_dis = ModflowGwfdis(fp3_gwf, nlay=1, nrow=2, ncol=2, top=1.0)

fp3_dis.nlay

We can see the value, but what did we get in return?

In [ ]:
type(fp3_dis.nlay)

To access the value, it's necessary to call `.get_data()`:

In [ ]:
fp3_dis.nlay.get_data()

These `MF...` data types are powerful, but when I'm building a model in Python, it's a bit distracting. And this one is misleading: `internal` is only a concept that applies to arrays.

On the other hand, package attributes in flopy4 are simple primitives, containers, and arrays. Serialization details are invisible unless you want them.

In [ ]:
dis = Dis(nlay=1, nrow=2, ncol=2, top=1.0)
dis.nlay

Another example: it's not always easy to determine the internal structure of record fields. Say we want to set rewetting options on an NPF package. We'd need to peruse the docs, or look at the `.dfn` property, to figure out what the rewet record consists of. We could look at the class docstring, but it's not always straightforward to interpret.

In [ ]:
from flopy.mf6 import ModflowGwfnpf

fp3_npf = ModflowGwfnpf(fp3_gwf, rewet_record=[1.0, 2, 2])
fp3_npf = ModflowGwfnpf(fp3_gwf, rewet_record=["wetfct", 1.0, "iwetit", 2, "ihdwet", 2])
fp3_npf.rewet_record

A NumPy recarray? This field only accepts a single record... Do we need to index into it to get the value?

In [ ]:
# fp3_npf.rewet_record[0]
# fp3_npf.rewet_record.get_data()  # closer...
fp3_npf.rewet_record.get_data()[0]  # got it

We've tried to make this easier in `flopy4` with first-class record classes. The API is self-describing, and intellisense works!

In [ ]:
from flopy4.mf6.gwf import Npf

npf = Npf(dims=dims, rewet=Npf.Rewet(wetfct=1.0, iwetit=2, ihdwet=2))
npf.rewet

In [ ]:
npf.rewet.wetfct

The `flopy4` prototype currently uses a library called [`attrs`](https://www.attrs.org/en/stable/), which makes it easy to convert POCOs (plain old Python classes) into standard library containers.

In [ ]:
from attrs import asdict, astuple
asdict(npf.rewet)
# astuple(npf.rewet)

**Note**: we're considering switching from `attrs` to [Pydantic](https://pydantic.dev/docs/validation/latest/get-started), but the API won't change much if at all &mdash; it may look like this instead: `npf.rewet.dict()`.

Let's build a model with `flopy4`!

In [ ]:
from pathlib import Path

import numpy as np

from flopy4.mf6 import Simulation, Ims
from flopy4.mf6.gwf import Gwf, Ic, Oc, Npf, Chd

sim_name = "flopy4_intro"
sim_ws = (Path("models") / sim_name).absolute()
sim_ws.mkdir(exist_ok=True, parents=True)
gwf_name = f"{sim_name}_gwf"

time = Time(perlen=[1.0], nstp=[1])
grid = StructuredGrid(
    nlay=1,
    nrow=10,
    ncol=10,
    delr=1.0 * np.ones(10),
    delc=1.0 * np.ones(10),
    top=1.0 * np.ones((10, 10)),
    botm=0.0 * np.ones((1, 10, 10)),
)
sim = Simulation(
    name=sim_name,
    workspace=sim_ws,
    tdis=time,  # you can pass `Time` in place of `Tdis`
)
ims = Ims(
    parent=sim,
    models=[gwf_name],
    outer_dvclose=1e-3,
    outer_maximum=25,
    inner_maximum=50,
    inner_dvclose=1e-3,
    rclose=Ims.Rclose(inner_rclose=0.1),
    linear_acceleration="cg",
)
gwf = Gwf(
    parent=sim,
    dis=grid,  # you can pass `Grid` in place of `Dis`
    name=gwf_name,
    save_flows=True
)
ic = Ic(parent=gwf, strt=1.0)
npf = Npf(parent=gwf, print_flows=True, save_flows=True, save_specific_discharge=True)
chd = Chd(
    parent=gwf,
    head={0: {(0, 0, 0): 1.0, (0, 9, 9): 0.0}},
)
oc = Oc(
    parent=gwf,
    budget_file=f"{gwf.name}.bud",
    head_file=f"{gwf.name}.hds",
    save_head={0: "all"},
    save_budget={0: "all"},
)

You can get an `xarray` view not only of standalone packages, but of models or simulations. In this case the return value is `xr.DataTree` instead of an `xr.Dataset`.

In [ ]:
sim.to_xarray()

Let's write and run it!

In [ ]:
sim.write()
sim.run(verbose=True)